In [1]:
cd(joinpath(pwd(), "src/LorentzianSimplexSolver"))

using Pkg
Pkg.activate(".")
Pkg.instantiate()
using LorentzianSimplexSolver

  Activating project at `~/Documents/Work/effective-spinfoam/code/Effective-Spinfoam/src/LorentzianSimplexSolver`


In [2]:
include("../../scripts/run_geometry.jl");
include("../../scripts/run_action.jl")
include("../../scripts/run_dlogEh_dg.jl")
include("../../scripts/run_djdl.jl")
include("../perturbations/TransverseBasis.jl")
include("../perturbations/LinearizedEOMs.jl")
include("../perturbations/QuadraticRegge.jl");

In [3]:
using JLD2

using .RunGeometry
using .RunAction
using .RunDlogEhDG
using .DJDLUtils
using .TransverseBasis
using .QuadraticRegge

#### Geometry setup

In [4]:
simplices = [[1,2,3,4,6],[1,2,3,5,6],[1,2,4,5,6],[1,3,4,5,6],[2,3,4,5,6]]

ns = length(simplices)

all_vertices = unique(Iterators.flatten(simplices))
sort!(all_vertices)

coords_lines = [
    "0, 0, 0, 0",
    "0, -2.7745276335252114, -0.9809436521275706, -1.6990442448471226",
    "0, 0, 0, -3.398088489694245",
    "-0.24028114141347542, -0.6936319083813028, -0.9809436521275706, -1.6990442448471226",
    "0, 0, -2.942830956382712, -1.6990442448471226",
    "-0.068,-0.27,-0.5,-1.3",
]

const ScalarT = Float64
# tol = 1e-10;
# const ScalarT = BigFloat
const tol = parse(ScalarT, "1e-8")

if ScalarT === BigFloat
    setprecision(BigFloat, 80)
    LorentzianSimplexSolver.PrecisionUtils.set_big_precision!(80)
    # LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(sqrt(eps(BigFloat)))
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
else
    LorentzianSimplexSolver.PrecisionUtils.set_tolerance!(tol)
end

gamma_vals = ScalarT(1);

vertex_coords = Dict{Int, Vector{ScalarT}}()  

for (v, line) in zip(all_vertices, coords_lines)
    vertex_coords[v] = LorentzianSimplexSolver.PrecisionUtils.parse_numeric_line(line, ScalarT)
end

In [5]:
geom_base = run_geometry_pipeline(simplices, coords_lines, ScalarT, tol);

#### Action and variables calculation

In [6]:
γ = LorentzianSimplexSolver.DefineAction.γsym()
sd_base, S_base = RunAction.run_action(geom_base, γ);

g_vars = geom_base.varias[:g_var]
z_vars = geom_base.varias[:z_var]
j_vars = geom_base.varias[:j_var]

vars = vcat(g_vars, z_vars, j_vars);

#### Hessian matrix computation or read Hessian from files

In [7]:
H_base = LorentzianSimplexSolver.EOMsHessian.compute_Hessian_block_half(S_base, vars);
H_base_eval = LorentzianSimplexSolver.EOMsHessian.evaluate_hessian_block(H_base, sd_base; γ=gamma_vals);

#### Computation of $\frac{\partial \log E_h}{\partial g_\alpha}$

In [8]:
g_vars = geom_base.varias[:g_var]
nh = length(geom_base.connectivity[1]["OrderBulkFaces"])

dlogEh_dg_sym = RunDlogEhDG.run_dlogEh_dg(geom_base)

dlogEh_dg_vals = RunDlogEhDG.evaluate_dlogEh_dg(dlogEh_dg_sym, geom_base, sd_base; γval=gamma_vals);

#### Computation of $\frac{\partial j_h}{\partial \ell_s}$

In [9]:
# djdl_matrix is a nh x nl matrix
djdl_matrix, bulk_edges, j_h_vertices = DJDLUtils.build_djdl_matrix(geom_base, vertex_coords, ScalarT, gamma_vals);
nl = length(bulk_edges)
nt = nh - nl;

#### Computation of $\hat{e}^i_h$

In [10]:
# eListHT is a nh x nt matrix
eListHT = TransverseBasis.compute_transverse_basis(djdl_matrix, tol);

#### computation of Linearized solution to EOMs

In [11]:
ng = length(g_vars)
nz = length(z_vars)
nx = ng + nz

HessianOld = H_base_eval
using Symbolics
@variables dl[1:nl]
dl_vec = collect(dl) 
dYsoln, HYY = LinearizedEOMs.solve_linearized_eoms(HessianOld, eListHT, djdl_matrix, nx, ScalarT, dl_vec);

#### Computation of $W_{h_1 h_2}$ matrix, $S_{ij}$ matrix and $\kappa_{h_1 h_2}$ matrix

In [12]:
invHessianXX = inv(HessianOld[1:nx, 1:nx])[1:ng, 1:ng];


Wmatrix = -dlogEh_dg_vals * invHessianXX * transpose(dlogEh_dg_vals);

Smatrix = eListHT' * Wmatrix * eListHT;
Smatrix_num = Complex{ScalarT}.(Symbolics.value.(Smatrix));

kappa_matrix = eListHT * inv(Smatrix_num) * eListHT';
quadratic_correct = - ScalarT(0.5) .* djdl_matrix' * Wmatrix * kappa_matrix * transpose(Wmatrix) * djdl_matrix;

In [13]:
W_all_matrix = -dlogEh_dg_vals * inv(HYY)[1:ng, 1:ng] * transpose(dlogEh_dg_vals);
Spinfoam_Quadratic1 = ScalarT(0.5) * transpose(djdl_matrix) * W_all_matrix * djdl_matrix

5×5 Matrix{ComplexF64}:
 -10.3619-0.28771im     -1.16749-0.0324164im   …    -2.8283-0.0785306im
 -1.16749-0.0324164im  -0.131541-0.00365238im     -0.318666-0.00884809im
 -7.19477-0.19977im    -0.810639-0.0225082im       -1.96382-0.0545275im
 -3.54319-0.0983802im  -0.399213-0.0110846im      -0.967117-0.026853im
  -2.8283-0.0785306im  -0.318666-0.00884809im     -0.771987-0.021435im

#### Computation of Quadratic term of Regge action $iS^{(2)}_{\mathrm{Regge}}$

In [14]:
dthetadl = QuadraticRegge.compute_dθDl(simplices, j_h_vertices, bulk_edges, vertex_coords, geom_base.connectivity[1]["Tets"], LorentzianSimplexSolver.Dihedral.minkowski_norm2, ScalarT);
dadl = gamma_vals .* djdl_matrix;
ReggeQuadratic = im * ScalarT(0.5) * dadl' * dthetadl;

#### Computation of Quadratic term correction $\frac{\gamma^2}{2} \kappa_{h_1 h_2}\delta\epsilon_{h_1}\,\delta\epsilon_{h_2}$ to $iS^{(2)}_{\mathrm{Regge}}$

In [15]:
correction = gamma_vals^2 * ScalarT(0.5) * dthetadl' * kappa_matrix * dthetadl;

#### Computation of Spinfoam Quadratic term $\mathcal I_{\mathrm{eff}}(\ell)-\mathring S$

In [16]:
Spinfoam_Quadratic = ReggeQuadratic + correction

5×5 Matrix{ComplexF64}:
 -10.3619-0.28771im     -1.16749-0.0324164im   …    -2.8283-0.0785306im
 -1.16749-0.0324164im  -0.131541-0.00365238im     -0.318666-0.00884809im
 -7.19477-0.19977im    -0.810639-0.0225082im       -1.96382-0.0545275im
 -3.54319-0.0983802im  -0.399213-0.0110846im      -0.967117-0.026853im
  -2.8283-0.0785306im  -0.318666-0.00884809im     -0.771987-0.021435im